In [ ]:
print("H")

In [ ]:
# Shared setup — imports, constants, paths
import os, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from scipy.fft import dctn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

# Backend-compatible constants — do not change
KYC_MAX_VIDEO_FRAMES = 5
KYC_FRAME_SIZE       = 224

ROOT            = Path("../../")
DATA_DIR        = ROOT / "data" / "kyc"
CROPS_DIR       = DATA_DIR / "crops"
WEIGHTS_DIR     = ROOT / "backend" / "weights"
PRECOMPUTED_DIR = ROOT / "backend" / "precomputed"
RESULTS_DIR     = Path("results")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Device:", DEVICE)

In [ ]:
# Video-level 80/10/10 split from pre-extracted face crops
# Crop filenames: <source>_<videostem>_f<N>.jpg

def build_split(label):
    crops = list((CROPS_DIR / label).glob("*.jpg"))
    groups = {}
    for p in crops:
        key = p.stem.rsplit("_", 1)[0]
        groups.setdefault(key, []).append(p)
    videos = list(groups.keys())
    train_v, temp_v = train_test_split(videos, test_size=0.2, random_state=SEED)
    val_v,   test_v = train_test_split(temp_v, test_size=0.5, random_state=SEED)
    return {s: [p for v in vs for p in groups[v]]
            for s, vs in [("train",train_v),("val",val_v),("test",test_v)]}

real_splits = build_split("real")
fake_splits = build_split("fake")

split_dfs = {}
for split in ("train", "val", "test"):
    rows = [(str(p), 0) for p in real_splits[split]] + \
           [(str(p), 1) for p in fake_splits[split]]
    split_dfs[split] = pd.DataFrame(rows, columns=["path", "label"])
    n  = len(split_dfs[split])
    nr = (split_dfs[split].label == 0).sum()
    nf = (split_dfs[split].label == 1).sum()
    print(f"{split:5s}: {n} samples  (real={nr}, fake={nf})")

test_paths = split_dfs["test"]["path"].values

In [ ]:
# ImageNet normalization — matches backend/services/kyc/inference.py exactly
IMAGENET_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
AUGMENT_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform or IMAGENET_TRANSFORM
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(Image.open(row["path"]).convert("RGB")), int(row["label"])

def make_loaders(batch_size=32):
    train_dl = DataLoader(FaceDataset(split_dfs["train"], AUGMENT_TRANSFORM),
                          batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = DataLoader(FaceDataset(split_dfs["val"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl  = DataLoader(FaceDataset(split_dfs["test"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl

In [ ]:
# DCT feature extractor — matches backend/services/kyc/frequency_analyzer.py exactly
def extract_dct_features(face_image, size=224):
    gray    = face_image.convert("L").resize((size, size))
    arr     = np.array(gray, dtype=np.float32) / 255.0
    dct     = dctn(arr, norm="ortho")
    log_mag = np.log1p(np.abs(dct))
    log_mag = (log_mag - log_mag.min()) / (log_mag.max() - log_mag.min() + 1e-8)
    return log_mag[np.newaxis, :, :]   # shape: (1, H, W)

class DCTDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dct = extract_dct_features(Image.open(row["path"]).convert("RGB"))
        return torch.tensor(dct, dtype=torch.float), int(row["label"])

def make_dct_loaders(batch_size=32):
    train_dl = DataLoader(DCTDataset(split_dfs["train"]), batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = DataLoader(DCTDataset(split_dfs["val"]),   batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl  = DataLoader(DCTDataset(split_dfs["test"]),  batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl

In [ ]:
# FrequencyCNN — matches backend/models/kyc/frequency_cnn.py exactly
class FrequencyCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),           nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),          nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

In [ ]:
# Training utilities — shared by all experiments

def train_one_epoch(model, loader, optimizer, criterion):
    model.train(); total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

def evaluate_loader(model, loader):
    """Return (labels, fake_probs) arrays for a full DataLoader."""
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for x, y in loader:
            prob = F.softmax(model(x.to(DEVICE)), dim=1)[:,1].cpu().numpy()
            all_probs.extend(prob); all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)

def train_model(model, train_dl, val_dl, optimizer, scheduler,
                epochs=20, save_path=None, patience=5):
    """Train with early stopping on val ROC-AUC."""
    criterion = nn.CrossEntropyLoss()
    best_auc, wait = 0.0, 0
    history = {"train_loss": [], "val_auc": []}
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_dl, optimizer, criterion)
        labels, probs = evaluate_loader(model, val_dl)
        val_auc = roc_auc_score(labels, probs)
        history["train_loss"].append(loss)
        history["val_auc"].append(val_auc)
        if scheduler: scheduler.step()
        print(f"Epoch {epoch:02d}/{epochs}  loss={loss:.4f}  val_auc={val_auc:.4f}")
        if val_auc > best_auc:
            best_auc, wait = val_auc, 0
            if save_path:
                torch.save(model.state_dict(), save_path)
                print(f"  -> Checkpoint saved (auc={best_auc:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}"); break
    return history

In [ ]:
# Standard 6-metric evaluation used by every experiment

def evaluate(y_true, y_pred_prob, threshold=0.5, title="Model"):
    y_pred = (np.array(y_pred_prob) >= threshold).astype(int)
    y_true = np.array(y_true)
    metrics = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_true, y_pred_prob),
        "PR-AUC":    average_precision_score(y_true, y_pred_prob),
    }
    print(f"\n--- {title} ---")
    for k, v in metrics.items(): print(f"  {k:12s}: {v:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)
    cm = confusion_matrix(y_true, y_pred)
    axes[0].imshow(cm, cmap="Blues")
    axes[0].set_title("Confusion Matrix")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, cm[i,j], ha="center", va="center", fontsize=14)
    axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
    axes[0].set_xticklabels(["Real","Fake"]); axes[0].set_yticklabels(["Real","Fake"])
    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    axes[1].plot(fpr, tpr, lw=2, label=f"AUC={metrics['ROC-AUC']:.3f}")
    axes[1].plot([0,1],[0,1],"--",color="grey")
    axes[1].set_title("ROC Curve"); axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
    axes[1].legend()
    prec, rec, _ = precision_recall_curve(y_true, y_pred_prob)
    axes[2].plot(rec, prec, lw=2, label=f"AP={metrics['PR-AUC']:.3f}")
    axes[2].set_title("PR Curve"); axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
    axes[2].legend()
    plt.tight_layout(); plt.show()
    return metrics

def threshold_analysis(y_true, y_pred_prob, title="Threshold Analysis"):
    thresholds = np.linspace(0.1, 0.9, 50)
    f1s, precs, recs = [], [], []
    for t in thresholds:
        yp = (np.array(y_pred_prob) >= t).astype(int)
        f1s.append(f1_score(y_true, yp, zero_division=0))
        precs.append(precision_score(y_true, yp, zero_division=0))
        recs.append(recall_score(y_true, yp, zero_division=0))
    plt.figure(figsize=(8,4))
    plt.plot(thresholds, f1s,   label="F1")
    plt.plot(thresholds, precs, label="Precision")
    plt.plot(thresholds, recs,  label="Recall")
    plt.axvline(0.50, color="orange", linestyle="--", label="suspicious (0.50)")
    plt.axvline(0.75, color="red",    linestyle="--", label="high_risk (0.75)")
    plt.title(title); plt.xlabel("Threshold"); plt.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Risk-tier assignment — matches backend/routers/kyc.py verdict_from_score exactly
def assign_tier(score):
    if score >= 0.75: return "high_risk"
    if score >= 0.50: return "suspicious"
    return "verified"

In [ ]:
# Experiment 2.5 — Model Complementarity + Ensemble
# Part A: Do the three models make different mistakes?
# Part B: Does simple average ensemble outperform each individual model?

# Load saved predictions from exp2.2, exp2.3, exp2.4
labels_eff  = np.load(RESULTS_DIR / "exp2.2_labels.npy")
probs_eff   = np.load(RESULTS_DIR / "exp2.2_probs.npy")
probs_vit   = np.load(RESULTS_DIR / "exp2.3_probs.npy")
probs_freq  = np.load(RESULTS_DIR / "exp2.4_probs.npy")

print(f"Test samples: {len(labels_eff)}")
print(f"Real: {(labels_eff==0).sum()}  Fake: {(labels_eff==1).sum()}")

In [ ]:
# Prediction correlation between models
scores_df = pd.DataFrame({
    "label":         labels_eff,
    "efficientnet":  probs_eff,
    "vit":           probs_vit,
    "frequency_cnn": probs_freq,
})
corr = scores_df[["efficientnet","vit","frequency_cnn"]].corr()
print("Prediction Correlation Matrix:")
print(corr.round(4))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(corr.columns, rotation=30, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(3):
    for j in range(3):
        ax.text(j,i,f"{corr.values[i,j]:.3f}",ha="center",va="center",fontsize=11)
plt.colorbar(im, ax=ax); ax.set_title("Score Correlation")
plt.tight_layout(); plt.show()

In [ ]:
# Agreement analysis at threshold 0.5
preds_eff  = (probs_eff  >= 0.5).astype(int)
preds_vit  = (probs_vit  >= 0.5).astype(int)
preds_freq = (probs_freq >= 0.5).astype(int)
n = len(labels_eff)

agree_all = ((preds_eff==preds_vit) & (preds_vit==preds_freq)).sum()
agree_two = (
    ((preds_eff==preds_vit) & (preds_vit!=preds_freq)) |
    ((preds_eff==preds_freq) & (preds_eff!=preds_vit)) |
    ((preds_vit==preds_freq) & (preds_vit!=preds_eff))
).sum()
print(f"All three agree         : {agree_all} ({100*agree_all/n:.1f}%)")
print(f"Two agree, one out      : {agree_two} ({100*agree_two/n:.1f}%)")
print(f"All disagree            : {n-agree_all-agree_two} ({100*(n-agree_all-agree_two)/n:.1f}%)")

In [ ]:
# Error overlap — unique errors per model
err_eff  = set(np.where(preds_eff  != labels_eff)[0])
err_vit  = set(np.where(preds_vit  != labels_eff)[0])
err_freq = set(np.where(preds_freq != labels_eff)[0])
print(f"EfficientNet errors     : {len(err_eff)}")
print(f"ViT errors              : {len(err_vit)}")
print(f"FrequencyCNN errors     : {len(err_freq)}")
print(f"Shared (all three)      : {len(err_eff & err_vit & err_freq)}")
print(f"Unique to EfficientNet  : {len(err_eff - err_vit - err_freq)}")
print(f"Unique to ViT           : {len(err_vit - err_eff - err_freq)}")
print(f"Unique to FrequencyCNN  : {len(err_freq - err_eff - err_vit)}")

categories = ["EfficientNet","ViT","FrequencyCNN","All 3 Shared"]
counts = [len(err_eff-err_vit-err_freq), len(err_vit-err_eff-err_freq),
          len(err_freq-err_eff-err_vit), len(err_eff&err_vit&err_freq)]
plt.figure(figsize=(8,4))
plt.bar(categories, counts, color=["steelblue","darkorange","green","red"])
plt.title("Unique Errors per Model"); plt.ylabel("Count")
plt.tight_layout(); plt.show()

In [ ]:
# Ensemble: simple average — matches backend/services/kyc/inference.py exactly
# ensemble = (efficientnet + vit + frequency_cnn) / 3
probs_ensemble = (probs_eff + probs_vit + probs_freq) / 3
metrics_ens = evaluate(labels_eff, probs_ensemble, title="Ensemble")

In [ ]:
# Load cumulative results from exp2.4 and add ensemble row
prev = pd.read_csv(RESULTS_DIR / "exp2.4_results.csv")
results = pd.concat([prev, pd.DataFrame([{"Model":"Ensemble",
    **{k:round(v,4) for k,v in metrics_ens.items()}}])], ignore_index=True)
results.to_csv(RESULTS_DIR / "exp2.5_results.csv", index=False)
print(results.to_string(index=False))

In [ ]:
# Grouped bar chart: all 5 models across all 6 metrics
metrics_cols = ["Accuracy","Precision","Recall","F1","ROC-AUC","PR-AUC"]
models = results["Model"].tolist()
x = np.arange(len(metrics_cols))
width = 0.16
fig, ax = plt.subplots(figsize=(14, 5))
colors = ["steelblue","darkorange","green","red","purple"]
for i, (model, color) in enumerate(zip(models, colors)):
    row = results[results["Model"]==model].iloc[0]
    ax.bar(x + i*width, [row[m] for m in metrics_cols], width, label=model, color=color)
ax.set_xticks(x + width*2); ax.set_xticklabels(metrics_cols)
ax.set_ylim(0, 1.05); ax.set_title("All Models — All Metrics")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Risk-tier analysis using production thresholds (matches router verdict_from_score)
tiers = [assign_tier(s) for s in probs_ensemble]
tier_df = pd.DataFrame({"tier":tiers,"label":labels_eff})
summary = tier_df.groupby("tier").agg(
    count=("label","count"),
    real=("label", lambda x: (x==0).sum()),
    fake=("label", lambda x: (x==1).sum()),
).reset_index()
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(probs_ensemble[labels_eff==0], bins=50, alpha=0.6, label="Real", color="steelblue")
ax.hist(probs_ensemble[labels_eff==1], bins=50, alpha=0.6, label="Fake", color="crimson")
ax.axvline(0.50, color="orange", linestyle="--", label="suspicious (0.50)")
ax.axvline(0.75, color="red",    linestyle="--", label="high_risk (0.75)")
ax.set_xlabel("Ensemble Score"); ax.set_title("Score Distribution with Risk Tiers")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Save ensemble scores for exp2.6 and precompute
np.save(RESULTS_DIR / "exp2.5_ensemble_probs.npy", probs_ensemble)
print("exp2.5 done")